# Полный пайплайн парсинга статей: Docling + Grobid (Colab / GPU)

Автономный ноутбук — извлекает из научных PDF всё сразу: текст, таблицы, формулы, картинки (Docling) И метаданные, расшифрованные ссылки и контексты цитирования (Grobid).

## Что извлекается
- Текст, заголовки, reading order, **таблицы** (с объединёнными ячейками — в markdown как HTML), **мат. формулы** (LaTeX на GPU), картинки — Docling
- **OCR** для сканированных/картиночных PDF — Docling (do_ocr)
- Метаданные статьи (title, authors, abstract, DOI, journal, publisher) — Grobid
- Расшифрованные ссылки (автор / журнал / год / DOI) — Grobid
- Контексты цитирования (какой маркер в тексте ведёт к какой ссылке) — Grobid

Внимание: **химические структуры** (2D-молекулы -> SMILES) НЕ извлекаются — для этого нужен отдельный OCSR-инструмент (DECIMER), здесь он отключён по умолчанию.

## Инструкция
1. Runtime -> Change runtime type -> T4 GPU -> Save
2. Запустите ячейки по порядку (Ячейка 2 — сборка Grobid, ~3-5 мин, один раз)
3. В Ячейке 5 загрузите PDF-файлы
4. Дождитесь двух этапов парсинга (Ячейки 7 и 9)
5. Скачайте ZIP (Ячейка 10)
6. В Streamlit -> «Парсинг статей» -> «Импорт из Colab» -> загрузите ZIP

## Структура результата
- stem.md              — Docling: текст+таблицы+формулы
- images/              — Docling: картинки
- metadata.json        — Grobid: метаданные статьи
- references.json      — Grobid: расшифрованные ссылки
- citation_contexts.json — Grobid: in-text -> reference
- stem.grobid.tei.xml  — сырой TEI (аудит)

In [ ]:
# ЯЧЕЙКА 1: Установка зависимостей
import os, subprocess, sys, shutil

# --- 1/3 OpenJDK 17 (для Grobid) ---
print("1/3 Установка OpenJDK 17 (для Grobid)...")
r = subprocess.run(["apt-get", "update", "-qq"])
if r.returncode != 0:
    raise RuntimeError("apt-get update завершился с ошибкой (код %d)" % r.returncode)
r = subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"])
if r.returncode != 0:
    raise RuntimeError("Не удалось установить OpenJDK (код %d)" % r.returncode)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
r = subprocess.run(["java", "-version"], stderr=subprocess.STDOUT)
if r.returncode != 0:
    raise RuntimeError("Java установилась, но java -version падает")
print("   Java готова")

# --- 2/3 Клонирование Grobid ---
print("2/3 Клонирование Grobid (release 0.8.1)...")
if os.path.exists("/content/grobid"):
    shutil.rmtree("/content/grobid")
r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", "0.8.1",
     "https://github.com/kermitt2/grobid.git", "/content/grobid"]
)
if r.returncode != 0:
    raise RuntimeError("Не удалось клонировать Grobid (проверьте интернет)")
print("   Grobid клонирован")

# --- 3/3 Python-зависимости ---
# ВАЖНО: easyocr нужен, т.к. в Ячейке 4 мы запрашиваем EasyOCR (ru + en). Раньше
# easyocr НЕ устанавливался, и converter.convert() падал с "EasyOCR is not installed",
# из-за чего НЕ сохранялись ни stem.md, ни картинки.
print("3/3 Установка Python-зависимостей...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "docling", "easyocr", "pypdf", "lxml", "requests", "Pillow"])
if r.returncode != 0:
    raise RuntimeError("pip install завершился с ошибкой (код %d)" % r.returncode)

# Жёсткая проверка базовых импортов (easyocr проверяем мягко — есть фолбэк в Ячейке 4)
print("   Проверка импортов...")
missing = []
for mod in ["docling", "pypdf", "lxml", "requests", "PIL"]:
    try:
        __import__(mod)
    except Exception as e:
        missing.append("%s (%s)" % (mod, e))
if missing:
    raise RuntimeError("Не импортируются: " + "; ".join(missing))
try:
    import easyocr  # noqa: F401
    print("   easyocr: готов")
except Exception as e_eocr:
    print("   ВНИМАНИЕ: easyocr не импортируется (%s)." % e_eocr)
    print("            OCR откатится на движок Docling по умолчанию (см. Ячейку 4).")
print("Все зависимости установлены и проверены!")


In [ ]:
# ЯЧЕЙКА 2: Сборка Grobid (~3-5 мин, один раз) + Heartbeat-хелпер для прогресса
import subprocess, threading, time

# Heartbeat: печатает прошедшее время каждые N секунд во время долгих операций.
# Переиспользуется в Ячейках 6, 9. Пример: with Heartbeat("парсю", 15): ...
class Heartbeat:
    def __init__(self, label="", interval=15):
        self.label = label
        self.interval = interval
        self._stop = threading.Event()
        self._thread = None
        self._t0 = 0.0

    def _tick(self):
        while not self._stop.wait(self.interval):
            print("   ... %s %dc" % (self.label, int(time.time() - self._t0)), flush=True)

    def __enter__(self):
        self._t0 = time.time()
        self._stop.clear()
        self._thread = threading.Thread(target=self._tick, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=2)
        return False


# Фикс известной проблемы Grobid 0.8.1 + Gradle 7.2: задача installDist падает с
# "Entry lib/langdetect-1.1-20120112.jar is a duplicate but no duplicate handling
#  strategy has been set". Решение — задать EXCLUDE для задач копирования через
# init-скрипт (дубли это один и тот же JAR, EXCLUDE ничего не теряет).
INIT_GRADLE = r"""import org.gradle.api.tasks.Copy
import org.gradle.api.tasks.Sync
import org.gradle.api.file.DuplicatesStrategy

allprojects {
    tasks.withType(Sync).configureEach {
        duplicatesStrategy = DuplicatesStrategy.EXCLUDE
    }
    tasks.withType(Copy).configureEach {
        duplicatesStrategy = DuplicatesStrategy.EXCLUDE
    }
}
"""
with open("/tmp/grobid_init.gradle", "w") as f:
    f.write(INIT_GRADLE)

print("Сборка Grobid (skip tests, стриминг вывода + heartbeat)...")
cmd = ["./gradlew", "--init-script", "/tmp/grobid_init.gradle",
       "clean", "installDist", "-x", "test", "--console=plain"]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd="/content/grobid",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
last_lines = []
with Heartbeat("собираю Grobid", interval=20):
    for line in proc.stdout:
        line = line.rstrip()
        if not line:
            continue
        last_lines.append(line)
        if len(last_lines) > 25:
            last_lines.pop(0)
        low = line.lower()
        if any(k in low for k in ("> task", "build successful", "build failed",
                                  "error", "exception", "installed", "distribution",
                                  "duplicate", "what went wrong")):
            print(line[:200], flush=True)
rc = proc.wait()
elapsed = int(time.time() - t0)
if rc != 0:
    print("\n=== ПОСЛЕДНИЕ СТРОКИ ВЫВОДА GRADLE ===")
    print("\n".join(last_lines))
    raise RuntimeError(
        "Сборка Grobid не удалась (код %d) за %dc.\n"
        "Смотрите строки выше — там указана причина (часто: дубль JAR / OOM / нет интернета)."
        % (rc, elapsed)
    )
print("Сборка завершена за %dc" % elapsed)

In [ ]:
# ЯЧЕЙКА 3: Запуск Grobid-сервера в фоне + проверка готовности
import subprocess, time, os, urllib.request

GROBID_URL = "http://localhost:8070"

os.makedirs("/content/grobid_logs", exist_ok=True)
log_f = open("/content/grobid_logs/service.log", "w")
grobid_proc = subprocess.Popen(
    ["/content/grobid/gradlew", "-p", "/content/grobid", "run"],
    stdout=log_f, stderr=subprocess.STDOUT,
    cwd="/content/grobid",
    start_new_session=True,
)
print("Grobid запускается (PID %d)..." % grobid_proc.pid)

# Модели грузятся ~60-120 сек при первом старте
ready = False
for i in range(60):
    try:
        with urllib.request.urlopen(GROBID_URL + "/api/isalive", timeout=3) as r:
            if r.read().strip() == b"true":
                ready = True
                break
    except Exception:
        pass
    time.sleep(5)
    if i % 6 == 0:
        print("  ...жду Grobid (%dc)" % ((i + 1) * 5))

if ready:
    print("Grobid готов:", GROBID_URL)
else:
    print("ВНИМАНИЕ: локальный Grobid не поднялся за 5 мин.")
    print("Переключаюсь на публичный demo-сервер Hugging Face Space.")
    GROBID_URL = "https://grobidOrg-grobid.hf.space"
    try:
        with urllib.request.urlopen(GROBID_URL + "/api/isalive", timeout=15) as r:
            demo_ok = (r.read().strip() == b"true")
    except Exception as e:
        demo_ok = False
        print("Demo-сервер тоже недоступен:", e)
    if demo_ok:
        print("Использую demo-сервер:", GROBID_URL)
        print("Внимание: у demo есть квоты/лимиты. Для партии PDF лучше свой Grobid.")
    else:
        print("ОШИБКА: ни локальный, ни demo Grobid недоступны. Лог: /content/grobid_logs/service.log")
print("GROBID_URL =", GROBID_URL)

In [ ]:
# ЯЧЕЙКА 4: Инициализация Docling (формулы + OCR + таблицы + GPU)
import time, logging, re, os, shutil, tempfile
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print("Загрузка модели Docling (формулы + OCR + таблицы)...")
_start = time.time()
try:
    from docling.document_converter import DocumentConverter
    from docling.datamodel.pipeline_options import ThreadedPdfPipelineOptions
    from docling.document_converter import FormatOption
    from docling.datamodel.base_models import InputFormat
    from docling.backend.docling_parse_backend import DoclingParseDocumentBackend
    from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

    # do_formula_enrichment — математика -> LaTeX (на GPU)
    pipeline_opts = ThreadedPdfPipelineOptions(do_formula_enrichment=True)

    # generate_picture_images=True выставляем БЕЗ "глотающего" try/except: без этого
    # пиксели картинок не сохраняются и _extract_images (Ячейка 6) не найдёт ни
    # одного изображения. Раньше строка лежала внутри `except Exception: pass` и
    # могла молча не примениться.
    pipeline_opts.generate_picture_images = True
    pipeline_opts.images_scale = 2.0

    # OCR для сканированных/картиночных PDF + структура таблиц
    pipeline_opts.do_ocr = True
    pipeline_opts.do_table_structure = True

    # OCR-движок: предпочитаем EasyOCR (ru + en), но ОБЯЗАТЕЛЬНО проверяем, что пакет
    # easyocr РЕАЛЬНО импортируется. EasyOcrOptions(...) строится успешно даже без
    # установленного easyocr — и тогда converter.convert() падал с
    # "EasyOCR is not installed", из-за чего НЕ сохранялись ни md, ни картинки.
    # Если easyocr недоступен — откатываемся на движок Docling по умолчанию, чтобы
    # конвертация НЕ падала.
    easyocr_importable = False
    try:
        import easyocr  # noqa: F401
        easyocr_importable = True
    except Exception as e_eimp:
        print("   EasyOCR не установлен/не импортируется (%s)" % e_eimp)

    if easyocr_importable:
        try:
            from docling.datamodel.pipeline_options import EasyOcrOptions
            pipeline_opts.ocr_options = EasyOcrOptions(lang=["ru", "en"])
            print("   OCR: EasyOCR (ru + en)")
        except Exception as e_ocr:
            print("   EasyOcrOptions недоступен (%s) — OCR на движке Docling по умолчанию" % e_ocr)
    else:
        print("   OCR: движок Docling по умолчанию (OcrAuto/RapidOCR).")
        print("        Для русского+английского через EasyOCR добавьте `easyocr` в pip install (Ячейка 1).")

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: FormatOption(
                pipeline_options=pipeline_opts,
                backend=DoclingParseDocumentBackend,
                pipeline_cls=StandardPdfPipeline,
            )
        }
    )
except Exception as e:
    print("ОШИБКА инициализации Docling:", e)
    print("Возможные причины: нет интернета для скачивания моделей, мало места на диске.")
    raise

gpu_name = "CPU"
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print("GPU:", gpu_name)
        print("   VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
    else:
        print("Внимание: GPU не обнаружен — парсинг будет медленнее")
except ImportError:
    print("Внимание: torch не установлен — GPU недоступен")

print("Модель загружена за %.1fc | Устройство: %s" % (time.time() - _start, gpu_name))


In [ ]:
# ЯЧЕЙКА 5: Загрузка PDF-файлов
from google.colab import files

for d in ["/content/uploaded", "/content/parsed"]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

print("Выберите PDF-файлы для парсинга...")
uploaded = files.upload()

pdf_files = []
for fname, data in uploaded.items():
    if fname.lower().endswith(".pdf"):
        path = "/content/uploaded/" + fname
        with open(path, "wb") as f:
            f.write(data)
        pdf_files.append((fname, path))
    else:
        print("  Пропуск: %s (не PDF)" % fname)

print("Загружено %d PDF-файлов" % len(pdf_files))
for fname, _ in pdf_files:
    print("  -", fname)

In [ ]:
# ЯЧЕЙКА 6: Функции Docling (текст+таблицы+формулы+картинки, рекурсивное деление при OOM)
import re, tempfile, time
from pathlib import Path

MAX_PAGES = 30  # PDF > 30 стр. делится пополам (рекурсивно, пока не уложится)


def _is_memory_error(exc):
    t = str(exc).lower()
    if "bad_alloc" in t or "bad allocation" in t:
        return True
    if isinstance(exc, MemoryError):
        return True
    return False


def _n_pages(pdf_path):
    try:
        from pypdf import PdfReader
        return len(PdfReader(str(pdf_path)).pages)
    except Exception:
        return 0


def _split_pdf_at(pdf_path, mid_page):
    from pypdf import PdfReader, PdfWriter
    reader = PdfReader(str(pdf_path))
    total = len(reader.pages)
    w1, w2 = PdfWriter(), PdfWriter()
    for i in range(mid_page):
        w1.add_page(reader.pages[i])
    for i in range(mid_page, total):
        w2.add_page(reader.pages[i])
    tmp = Path(tempfile.mkdtemp(prefix="docling_split_"))
    p1, p2 = tmp / "part1.pdf", tmp / "part2.pdf"
    with open(p1, "wb") as f:
        w1.write(f)
    with open(p2, "wb") as f:
        w2.write(f)
    return p1, p2


def _cleanup(p1, p2):
    try:
        for p in (p1, p2):
            if p.exists():
                p.unlink()
        if p1.parent.exists() and not list(p1.parent.iterdir()):
            p1.parent.rmdir()
    except Exception:
        pass


_LATEX_RE = re.compile(r"\$\$(.+?)\$\$", re.DOTALL)
_LATEX_INLINE_RE = re.compile(r"(?<!\$)\$(?!\$)(.+?)(?<!\$)\$(?!\$)")


def _count_latex(text):
    return len(_LATEX_RE.findall(text)) + len(_LATEX_INLINE_RE.findall(text))


def _iter_pictures(doc):
    # Канонический способ Docling — iterate_items() + фильтр по PictureItem.
    # Фолбэк на doc.pictures на случай старых версий.
    try:
        from docling_core.types.doc import PictureItem
        return [el for el, _ in doc.iterate_items() if isinstance(el, PictureItem)]
    except Exception:
        return list(getattr(doc, "pictures", []) or [])


def _extract_images(conv_result, images_dir, scale=2.0):
    count = 0
    # Картинки хранятся на conv_result.document, а не на самом conv_result
    doc = conv_result.document
    pictures = _iter_pictures(doc)
    # Сквозная нумерация: при рекурсивном делении PDF обе части пишут в ОДНУ папку,
    # поэтому продолжаем нумерацию с учётом уже сохранённых файлов — иначе вторая
    # половина затрёт image_001.png первой половины (потеря картинок).
    start = 1
    try:
        nums = []
        for p in images_dir.glob("image_*.png"):
            tail = p.stem.split("_")[-1]
            try:
                nums.append(int(tail))
            except ValueError:
                pass
        if nums:
            start = max(nums) + 1
    except Exception:
        pass
    for i, pic in enumerate(pictures, start=start):
        try:
            img = None
            if hasattr(pic, "get_image"):
                try:
                    # Docling v2 API: get_image(DoclingDocument) -> PIL.Image
                    img = pic.get_image(doc)
                except Exception:
                    pass
            # Фолбэк: картинка уже лежит как атрибут (некоторые версии Docling)
            if img is None and hasattr(pic, "image") and pic.image is not None:
                raw = pic.image
                img = getattr(raw, "pil_image", raw)  # ImageRef хранит PIL внутри
            if img is not None:
                img.save(str(images_dir / ("image_%03d.png" % i)), "PNG")
                count += 1
        except Exception as e:
            print("  Предупреждение: картинка %d не извлечена: %s" % (i, e))
    return count


def _convert_single(converter, pdf_path, images_dir=None, image_scale=2.0):
    # heartbeat вокруг самого долгого вызова, чтобы было видно, что не зависло
    with Heartbeat("парсю Docling", interval=15):
        result = converter.convert(str(pdf_path))
    md = result.document.export_to_markdown()
    placeholder_count = md.count("<!-- image -->")
    formulas_nd = md.count("<!-- formula-not-decoded -->")
    img_idx = 0
    out = []
    for line in md.split("\n"):
        if "<!-- image -->" in line:
            img_idx += 1
            line = line.replace("<!-- image -->", "[IMAGE_%d]" % img_idx)
        out.append(line)
    md = "\n".join(out)
    md = md.replace("<!-- formula-not-decoded -->", "")
    formulas = _count_latex(md)
    n_imgs = 0
    if images_dir is not None:
        n_imgs = _extract_images(result, images_dir, image_scale)
    return md, placeholder_count, formulas, formulas_nd, n_imgs, False


def _merge_results(parts):
    md = "\n\n".join(p[0] for p in parts if p)
    ph = sum(p[1] for p in parts if p)
    fm = sum(p[2] for p in parts if p)
    fm_nd = sum(p[3] for p in parts if p)
    ni = sum(p[4] for p in parts if p)
    return md, ph, fm, fm_nd, ni, True


def _process_part(converter, part_path, images_dir, image_scale, max_pages, depth=0):
    # Рекурсивно: если страниц > max_pages -> дробим пополам; при OOM тоже дробим.
    n = _n_pages(part_path)
    indent = "    " * (depth + 1)
    if max_pages > 0 and n > max_pages and n >= 2:
        mid = n // 2
        print("%s %d стр. > %d -> дроблю пополам" % (indent, n, max_pages))
        p1, p2 = _split_pdf_at(part_path, mid)
        try:
            r1 = _process_part(converter, p1, images_dir, image_scale, max_pages, depth + 1)
            r2 = _process_part(converter, p2, images_dir, image_scale, max_pages, depth + 1)
            return _merge_results([r1, r2])
        finally:
            _cleanup(p1, p2)
    try:
        return _convert_single(converter, part_path, images_dir, image_scale)
    except Exception as e:
        if _is_memory_error(e) and n >= 2:
            mid = n // 2
            print("%s нехватка памяти -> дроблю пополам" % indent)
            p1, p2 = _split_pdf_at(part_path, mid)
            try:
                r1 = _process_part(converter, p1, images_dir, image_scale, max_pages, depth + 1)
                r2 = _process_part(converter, p2, images_dir, image_scale, max_pages, depth + 1)
                return _merge_results([r1, r2])
            except Exception as e2:
                print("%s часть пропущена: %s" % (indent, e2))
                return ("<!-- часть пропущена: %s -->" % e2, 0, 0, 0, 0, True)
            finally:
                _cleanup(p1, p2)
        raise


def parse_pdf_docling(converter, pdf_path, output_dir, extract_images_flag=True,
                      image_scale=2.0, max_pages=MAX_PAGES):
    pdf_path = Path(pdf_path)
    stem = pdf_path.stem
    article_dir = Path(output_dir) / stem
    article_dir.mkdir(parents=True, exist_ok=True)
    images_dir = article_dir / "images" if extract_images_flag else None
    if images_dir:
        images_dir.mkdir(parents=True, exist_ok=True)
    big = _n_pages(pdf_path) > max_pages
    try:
        md, ph, fm, fm_nd, n_imgs, split_used = _process_part(
            converter, pdf_path, images_dir, image_scale, max_pages)
        (article_dir / ("%s.md" % stem)).write_text(md, encoding="utf-8")
        return {"stem": stem, "chars": len(md), "placeholders": ph, "formulas": fm,
                "formulas_nd": fm_nd, "images": n_imgs, "split_used": split_used or big}
    except Exception as e:
        return {"stem": stem, "error": str(e)}


print("Функции Docling определены (MAX_PAGES=%d)" % MAX_PAGES)


In [ ]:
# ЯЧЕЙКА 7: ЭТАП 1 — Docling парсинг (текст, таблицы, формулы LaTeX, картинки)
import time

docling_results = []
t_total = time.time()
total = len(pdf_files)

for i, (fname, pdf_path) in enumerate(pdf_files, 1):
    print("\n[Этап 1] [%d/%d] %s" % (i, total, fname))
    t0 = time.time()
    res = parse_pdf_docling(converter, pdf_path, "/content/parsed",
                            extract_images_flag=True, image_scale=2.0)
    res["docling_time"] = round(time.time() - t0, 1)
    docling_results.append(res)
    if "error" in res:
        print("  ОШИБКА:", res["error"])
    else:
        split_tag = " (разделён)" if res.get("split_used") else ""
        print("  OK: %d симв, %d табл-заглушек, %d формул, %d картинок, %dc%s" % (
            res["chars"], res["placeholders"], res["formulas"], res["images"],
            res["docling_time"], split_tag))

ok = sum(1 for r in docling_results if "error" not in r)
print("\n" + "=" * 60)
print("Этап 1 (Docling) итог: %d/%d успешно за %.1fc" % (ok, total, time.time() - t_total))
print("=" * 60)

In [ ]:
# ЯЧЕЙКА 8: Функции Grobid — отправка PDF + разбор TEI -> метаданные/ссылки/контексты
import json as _json
import time
import requests as _requests
from requests.exceptions import Timeout, ConnectionError as ReqConnError
from lxml import etree as _etree
from pathlib import Path

TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}
TEI = "{http://www.tei-c.org/ns/1.0}"
XMLID = "{http://www.w3.org/XML/1998/namespace}id"


def _text(el):
    if el is None:
        return ""
    return " ".join("".join(el.itertext()).split())


def _authors(author_els):
    out = []
    for a in author_els:
        pn = a.find("tei:persName", TEI_NS)
        if pn is None:
            r = _text(a)
            if r:
                out.append({"raw": r})
            continue
        fore = [t.text or "" for t in pn.findall("tei:forename", TEI_NS)]
        sur_el = pn.find("tei:surname", TEI_NS)
        sur = sur_el.text if sur_el is not None and sur_el.text else ""
        out.append({"forename": " ".join(fore).strip(), "surname": sur.strip()})
    return out


def _ref_to_dict(bibl, ref_id):
    d = {"ref_id": ref_id}
    analytic = bibl.find("tei:analytic", TEI_NS)
    monogr = bibl.find("tei:monogr", TEI_NS)
    title = None
    if analytic is not None:
        t = analytic.find("tei:title[@type='main']", TEI_NS) or analytic.find("tei:title", TEI_NS)
        if t is not None:
            title = _text(t)
    if not title and monogr is not None:
        jt = monogr.find("tei:title[@level='j']", TEI_NS)
        if jt is not None:
            title = _text(jt)
    d["title"] = title
    authors_src = analytic if analytic is not None else monogr
    if authors_src is not None:
        au = _authors(authors_src.findall("tei:author", TEI_NS))
        if au:
            d["authors"] = au
    if monogr is not None:
        jt = monogr.find("tei:title[@level='j']", TEI_NS)
        if jt is not None:
            d["journal"] = _text(jt)
        date = monogr.find("tei:imprint/tei:date", TEI_NS)
        if date is not None:
            when = date.get("when")
            if when:
                d["year"] = when[:4]
            else:
                d["year"] = _text(date)
        imprint = monogr.find("tei:imprint", TEI_NS)
        if imprint is not None:
            pub = imprint.find("tei:publisher", TEI_NS)
            if pub is not None:
                d["publisher"] = _text(pub)
            for bs in imprint.findall("tei:biblScope", TEI_NS):
                unit = bs.get("unit")
                val = _text(bs) or bs.get("to") or bs.get("from")
                if unit and val:
                    d[unit] = val
    for idno in bibl.iter(TEI + "idno"):
        itype = idno.get("type")
        val = _text(idno)
        if itype == "DOI" and val:
            d["doi"] = val
        elif itype == "PMID" and val:
            d["pmid"] = val
        elif itype == "URL" and val:
            d["url"] = val
    note = bibl.find(".//tei:note", TEI_NS)
    d["raw"] = _text(note) if note is not None else None
    return d


def parse_tei(tei_bytes):
    root = _etree.fromstring(tei_bytes)
    ns = TEI_NS
    meta = {}
    src = root.find(".//tei:sourceDesc/tei:biblStruct", ns)
    if src is not None:
        analytic = src.find("tei:analytic", ns)
        monogr = src.find("tei:monogr", ns)
        # Заголовок статьи (level='a' обычно в analytic)
        t = None
        if analytic is not None:
            t = analytic.find("tei:title[@type='main']", ns) or analytic.find("tei:title", ns)
        if t is None and monogr is not None:
            t = monogr.find("tei:title[@level='j']", ns) or monogr.find("tei:title", ns)
        if t is not None:
            meta["title"] = _text(t)
        # Авторы статьи — из analytic (стандарт TEI), фолбэк monogr
        authors_src = analytic if analytic is not None else monogr
        if authors_src is not None:
            au = _authors(authors_src.findall("tei:author", ns))
            if au:
                meta["authors"] = au
        if monogr is not None:
            jt = monogr.find("tei:title[@level='j']", ns)
            if jt is not None:
                meta["journal"] = _text(jt)
            date = monogr.find("tei:imprint/tei:date", ns)
            if date is not None and date.get("when"):
                meta["date"] = date.get("when")
            pub = monogr.find("tei:imprint/tei:publisher", ns)
            if pub is not None:
                meta["publisher"] = _text(pub)
        for idno in src.iter(TEI + "idno"):
            if idno.get("type") == "DOI":
                meta["doi"] = _text(idno)
    abstr = root.find(".//tei:profileDesc/tei:abstract", ns)
    if abstr is not None:
        meta["abstract"] = _text(abstr)
    kws = root.findall(".//tei:keywords/tei:term", ns)
    if kws:
        meta["keywords"] = [_text(k) for k in kws]

    refs = []
    for bibl in root.findall(".//tei:back//tei:listBibl/tei:biblStruct", ns):
        rid = bibl.get(XMLID) or ("b%d" % len(refs))
        refs.append(_ref_to_dict(bibl, rid))

    contexts = []
    body = root.find(".//tei:body", ns)
    if body is not None:
        for ref in body.iter(TEI + "ref"):
            if ref.get("type") != "bibr":
                continue
            tgt = (ref.get("target") or "").lstrip("#")
            marker = _text(ref) or (ref.text or "").strip()
            parent_p = ref.getparent()
            context_text = ""
            if parent_p is not None:
                full = _text(parent_p)
                pos = full.find(marker) if marker else -1
                if 0 <= pos and len(full) > 500:
                    start = max(0, pos - 200)
                    context_text = full[start:start + 400]
                else:
                    context_text = full[:400]
            if tgt:
                contexts.append({"ref_id": tgt, "marker": marker, "context": context_text})

    return {"metadata": meta, "references": refs, "citation_contexts": contexts}


def process_with_grobid(pdf_path, grobid_url, consolidate=2, timeout=180, retries=2):
    # Отправить PDF в Grobid, вернуть TEI-байты. Различает типы ошибок,
    # ретраит при 503 (перегрузка) с backoff.
    url = grobid_url + "/api/processFulltextDocument"
    last_err = None
    for attempt in range(1, retries + 2):
        try:
            with open(pdf_path, "rb") as f:
                resp = _requests.post(
                    url,
                    files={"input": (Path(pdf_path).name, f, "application/pdf")},
                    data={"consolidate": str(consolidate)},
                    timeout=timeout,
                )
        except Timeout:
            last_err = RuntimeError("Grobid не ответил за %dc (timeout). "
                                    "PDF слишком большой или сервер перегружен." % timeout)
            break
        except ReqConnError as e:
            last_err = RuntimeError("Нет связи с Grobid (%s). Сервер упал?" % e)
            break
        except Exception as e:
            last_err = RuntimeError("Сетевая ошибка при обращении к Grobid: %s" % e)
            break
        if resp.status_code == 200:
            return resp.content
        if resp.status_code == 503:
            wait = 15 * attempt
            print("  Grobid перегружен (503), жду %dc и повторяю (%d/%d)..." % (
                wait, attempt, retries + 1))
            time.sleep(wait)
            last_err = RuntimeError("Grobid перегружен (503) после %d попыток." % (retries + 1))
            continue
        if resp.status_code == 500:
            snippet = (resp.text or "")[:300]
            last_err = RuntimeError("Grobid внутренняя ошибка (500). Возможно битый PDF. Фрагмент: %s" % snippet)
        elif resp.status_code == 413:
            last_err = RuntimeError("PDF слишком большой для Grobid (413 Payload Too Large).")
        elif resp.status_code == 400:
            last_err = RuntimeError("Grobid не принял PDF (400): %s" % (resp.text or "")[:300])
        else:
            last_err = RuntimeError("Grobid вернул HTTP %d: %s" % (
                resp.status_code, (resp.text or "")[:200]))
        break
    raise last_err


print("Функции Grobid определены")

In [ ]:
# ЯЧЕЙКА 9: ЭТАП 2 — Grobid (метаданные + расшифрованные ссылки + контексты цитирования)
import time, json

grobid_results = []
t_total = time.time()
total = len(pdf_files)

for i, (fname, pdf_path) in enumerate(pdf_files, 1):
    stem = Path(fname).stem
    print("\n[Этап 2] [%d/%d] %s" % (i, total, fname))
    t0 = time.time()
    article_dir = Path("/content/parsed") / stem
    article_dir.mkdir(parents=True, exist_ok=True)
    try:
        print("  -> отправляю в Grobid (consolidate=2, подбор DOI через Crossref)...")
        with Heartbeat("жду Grobid", interval=20):
            tei_bytes = process_with_grobid(pdf_path, GROBID_URL, consolidate=2, timeout=180)
        (article_dir / ("%s.grobid.tei.xml" % stem)).write_bytes(tei_bytes)
        parsed = parse_tei(tei_bytes)
        meta = parsed["metadata"]
        refs = parsed["references"]
        ctxs = parsed["citation_contexts"]
        with open(article_dir / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        with open(article_dir / "references.json", "w", encoding="utf-8") as f:
            json.dump(refs, f, ensure_ascii=False, indent=2)
        with open(article_dir / "citation_contexts.json", "w", encoding="utf-8") as f:
            json.dump(ctxs, f, ensure_ascii=False, indent=2)
        n_doi = sum(1 for r in refs if r.get("doi"))
        gt = time.time() - t0
        meta_title = (meta.get("title") or "")[:60]
        n_authors = len(meta.get("authors", []))
        print("  OK: метаданные (%s...), %d авторов, %d ссылок (%d с DOI), %d контекстов, %.1fc" % (
            meta_title, n_authors, len(refs), n_doi, len(ctxs), gt))
        grobid_results.append({"stem": stem, "references": len(refs),
                               "with_doi": n_doi, "contexts": len(ctxs),
                               "grobid_time": round(gt, 1)})
    except Exception as e:
        print("  ОШИБКА Grobid:", e)
        grobid_results.append({"stem": stem, "error": str(e)})

ok = sum(1 for r in grobid_results if "error" not in r)
total_refs = sum(r.get("references", 0) for r in grobid_results)
total_doi = sum(r.get("with_doi", 0) for r in grobid_results)
print("\n" + "=" * 60)
print("Этап 2 (Grobid) итог: %d/%d успешно за %.1fc" % (ok, total, time.time() - t_total))
print("Всего расшифровано ссылок: %d (с DOI: %d)" % (total_refs, total_doi))
print("=" * 60)

In [ ]:
# ЯЧЕЙКА 10: Отчёт + ZIP + скачивание
import zipfile
from datetime import datetime
from google.colab import files as colab_files


def _build_report():
    lines = ["# Отчёт парсинга: Docling + Grobid (Colab GPU)", ""]
    lines.append("## Этап 1: Docling (текст, таблицы, формулы, картинки)")
    for r in docling_results:
        stem = r.get("stem", "?")
        if "error" in r:
            lines.append("- X %s: ОШИБКА %s" % (stem, r["error"]))
        else:
            split_tag = " (разделён)" if r.get("split_used") else ""
            lines.append("- OK %s: %d симв, %d формул, %d картинок, %dc%s" % (
                stem, r["chars"], r["formulas"], r["images"], r.get("docling_time", 0), split_tag))
    lines.append("")
    lines.append("## Этап 2: Grobid (метаданные, ссылки, контексты)")
    for r in grobid_results:
        stem = r.get("stem", "?")
        if "error" in r:
            lines.append("- X %s: ОШИБКА %s" % (stem, r["error"]))
        else:
            lines.append("- OK %s: %d ссылок (%d с DOI), %d контекстов, %dc" % (
                stem, r["references"], r["with_doi"], r["contexts"], r.get("grobid_time", 0)))
    lines.append("")
    return "\n".join(lines)


try:
    report_text = _build_report()
    with open("/content/parsed/_parsing_report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)
    print("Отчёт сохранён в /content/parsed/_parsing_report.txt")

    zip_name = "docling_grobid_parsed_%s.zip" % datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = "/content/" + zip_name
    n_files = 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, fnames in os.walk("/content/parsed"):
            for fn in fnames:
                fp = os.path.join(root, fn)
                arcname = os.path.relpath(fp, "/content")
                zf.write(fp, arcname)
                n_files += 1
    zip_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print("ZIP: %s (%.1f МБ, %d файлов)" % (zip_name, zip_mb, n_files))
    print("parsed/ статьи:")
    for r in docling_results:
        if "error" not in r:
            print("  parsed/%s/" % r["stem"])
    print("Скачивание...")
    colab_files.download(zip_path)
except Exception as e:
    print("ОШИБКА при сборке/скачивании ZIP:", e)
    print("Файлы остались в /content/parsed/ — скачайте вручную через панель Files слева.")

## Готово!

1. ZIP скачался на ваш компьютер
2. Откройте Streamlit -> «Парсинг статей»
3. Внизу страницы — секция «Импорт из Colab»
4. Загрузите ZIP -> файлы распакуются в data/articles/parsed/

В каждой папке статьи теперь:
- stem.md — текст с формулами LaTeX и таблицами (Docling)
- images/ — вырезанные картинки (Docling)
- metadata.json — метаданные статьи, включая авторов и издателя (Grobid)
- references.json — расшифрованные ссылки (Grobid)
- citation_contexts.json — маппинг in-text -> ссылка (Grobid)
- stem.grobid.tei.xml — сырой TEI для аудита

Если были ошибки — сообщите, приложив вывод проблемной ячейки (в отчёте и логах теперь видна причина).